# 📅 smartbuy: Category Seasonality Analysis

**Challenge 1 · Business Analyst, Projects & Operations**

smartbuy's Content Department identifies the seasonality of its product categories **manually, by eyeballing charts**.
This notebook replaces that with a repeatable analysis: it measures how seasonal each of the **100 categories** is,
when it peaks, and which categories should be planned for first.

The same logic powers the Streamlit web app (`streamlit run app.py`).

---

### Contents
1. Setup
2. Load and combine the data
3. Data quality and cleaning
4. Seasonality index and KPIs
5. KPIs overview
6. Top n categories by clicks
7. It's all about seasonality
8. Key findings and recommendations

## 1. Setup

### 1.1 Libraries
- **pandas / numpy** for data handling
- **plotly express** for interactive charts (the same charts as in the web app)
- **seaborn / matplotlib** for the static heatmap

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

### 1.2 Constants
Month names for readable charts, the mapping from month to season, and one fixed colour per season and per priority
group so every chart reads the same way.

In [ ]:
DATA_DIR = Path("../data")

MONTH_NAMES = {1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr", 5: "May", 6: "Jun",
               7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"}

SEASON_MAP = {12: "winter", 1: "winter", 2: "winter",
              3: "spring", 4: "spring", 5: "spring",
              6: "summer", 7: "summer", 8: "summer",
              9: "autumn", 10: "autumn", 11: "autumn"}

SEASON_ORDER = ["winter", "spring", "summer", "autumn", "no clear season"]
SEASON_COLORS = {"winter": "#3b82f6", "spring": "#22c55e", "summer": "#f59e0b",
                 "autumn": "#ea580c", "no clear season": "#94a3b8"}
PRIORITY_COLORS = {"1. big & seasonal": "#6366f1", "2. small & seasonal": "#a78bfa",
                   "3. big & steady": "#14b8a6", "4. small & steady": "#cbd5e1"}

## 2. Load and combine the data

### 2.1 Look at one file
The raw data comes as **100 CSV files, one per category** (`pis_1.csv` … `pis_100.csv`).
Before combining them we open one file to understand its structure.

In [ ]:
sample = pd.read_csv(DATA_DIR / "pis_1.csv")
print("shape:", sample.shape)
print(sample.dtypes)
sample.head()

**Columns**

| column | meaning |
|---|---|
| `category` | name of the category |
| `month` | month number (1–12) |
| `year` | year (2021) |
| `clicks` | clicks on the category in that month |

### 2.2 Combine all 100 files
All files share the same structure, so we read them and stack them below each other into one DataFrame.

In [ ]:
files = [DATA_DIR / f"pis_{i}.csv" for i in range(1, 101)]
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
df.info()

### 2.3 Check the result
We expect **1,200 rows** (100 categories × 12 months), exactly 12 rows per category and no duplicates.

In [ ]:
print("rows:              ", len(df))
print("categories:        ", df["category"].nunique())
print("rows per category: ", df.groupby("category").size().unique())
print("years:             ", df["year"].unique())
print("duplicate rows:    ", df.duplicated(["category", "month", "year"]).sum())

### 2.4 Save the combined table
The combined file is reused by the web app, so nobody has to combine the 100 files again.
`lineterminator="\n"` keeps the same line endings on Windows and Mac, so the file does not change when the notebook is re-run.

In [ ]:
df.to_csv(DATA_DIR / "combined_categories.csv", index=False, lineterminator="\n")

## 3. Data quality and cleaning

### 3.1 Missing values
Some months have no click value. We count them and look at which categories are affected.

In [ ]:
print("missing click values:", df["clicks"].isna().sum())
df[df["clicks"].isna()].groupby("category").size().sort_values(ascending=False).head(10)

### 3.2 Wide table: one row per category, one column per month
For the seasonality analysis it is easier to have the 12 months side by side.

In [ ]:
monthly_raw = df.pivot(index="category", columns="month", values="clicks")
monthly_raw.head()

### 3.3 Fill the missing months
A missing month is estimated from its **neighbouring months** (linear interpolation).

The year is treated as a **circle**: December and January are neighbours. To do that, the 12 months are written
three times in a row, interpolated, and the middle copy is kept. This way a missing January can use December,
and a missing December can use January.

In [ ]:
def fill_circular(row):
    tripled = pd.concat([row, row, row], ignore_index=True)
    filled = tripled.interpolate()
    return pd.Series(filled.iloc[12:24].values, index=row.index)


monthly = monthly_raw.apply(fill_circular, axis=1)
print("missing values before:", monthly_raw.isna().sum().sum())
print("missing values after: ", monthly.isna().sum().sum())

## 4. Seasonality index and KPIs

### 4.1 Seasonality index
Categories have very different sizes, so raw clicks cannot be compared directly.
The **seasonality index** divides every month by the category's average month:

$$\text{index}_{month} = \frac{\text{clicks}_{month}}{\text{average monthly clicks of the category}}$$

- **1.0** = an average month
- **1.5** = 50 % more clicks than average
- **0.5** = half the clicks of an average month

In [ ]:
seasonal = monthly.div(monthly.mean(axis=1), axis=0)
seasonal.round(2).head()

### 4.2 KPIs per category

| KPI | definition |
|---|---|
| `seasonality_score` | highest minus lowest monthly index: how strongly the category swings |
| `peak_month` / `low_month` | month with the highest / lowest index |
| `avg_monthly_clicks` | size of the category |
| `peak_vs_low` | clicks in the best month ÷ clicks in the weakest month |
| `filled_values` | number of months that were estimated |

In [ ]:
kpis = pd.DataFrame({
    "seasonality_score": seasonal.max(axis=1) - seasonal.min(axis=1),
    "peak_month": seasonal.idxmax(axis=1),
    "low_month": seasonal.idxmin(axis=1),
    "avg_monthly_clicks": monthly.mean(axis=1),
    "peak_vs_low": monthly.max(axis=1) / monthly.min(axis=1),
    "filled_values": monthly_raw.isna().sum(axis=1),
}).round(2)
kpis.sort_values("seasonality_score", ascending=False).head()

### 4.3 Seasonality strength and peak season
The score is split into three groups:

- **flat**: score up to 0.5 → no clear season
- **moderate**: 0.5 to 1.0
- **strong**: above 1.0

Every category also gets the season of its peak month. Flat categories get *no clear season*.

In [ ]:
kpis["strength"] = pd.cut(kpis["seasonality_score"], bins=[0, 0.5, 1.0, np.inf],
                          labels=["flat", "moderate", "strong"]).astype(str)
kpis["peak_season"] = kpis["peak_month"].map(SEASON_MAP)
kpis.loc[kpis["strength"] == "flat", "peak_season"] = "no clear season"
kpis["strength"].value_counts()

### 4.4 Priority groups
Two questions matter for planning: *Is the category big?* and *Is it seasonal?*
We split both at the **median** and get four groups:

| | seasonal | steady |
|---|---|---|
| **big** | 1. big & seasonal ⭐ | 3. big & steady |
| **small** | 2. small & seasonal | 4. small & steady |

In [ ]:
is_big = kpis["avg_monthly_clicks"] >= kpis["avg_monthly_clicks"].median()
is_seasonal = kpis["seasonality_score"] >= kpis["seasonality_score"].median()
kpis["priority"] = np.select(
    [is_big & is_seasonal, ~is_big & is_seasonal, is_big & ~is_seasonal],
    ["1. big & seasonal", "2. small & seasonal", "3. big & steady"],
    default="4. small & steady",
)
kpis["priority"].value_counts().sort_index()

### 4.5 Long table for the charts
Plotly works best with one row per category and month, so we build a long version of the data.

In [ ]:
monthly_long = (monthly.stack().rename("clicks").to_frame()
                .join(seasonal.stack().rename("index"))
                .reset_index())
monthly_long["month_name"] = monthly_long["month"].map(MONTH_NAMES)
monthly_long = monthly_long.merge(kpis[["peak_season", "strength", "priority"]],
                                  left_on="category", right_index=True)
monthly_long.head()

## 5. KPIs overview

### 5.1 Headline numbers

In [ ]:
month_share = monthly.sum() / monthly.sum().sum() * 100
peak_counts = kpis["peak_month"].value_counts()

headline = pd.Series({
    "categories": len(kpis),
    "total clicks 2021": f"{monthly.values.sum():,.0f}",
    "strongly seasonal": (kpis["strength"] == "strong").sum(),
    "busiest month": f"{MONTH_NAMES[month_share.idxmax()]} ({month_share.max():.1f}% of clicks)",
    "quietest month": f"{MONTH_NAMES[month_share.idxmin()]} ({month_share.min():.1f}% of clicks)",
    "most common peak month": f"{MONTH_NAMES[peak_counts.idxmax()]} ({peak_counts.max()} categories)",
    "average seasonality score": round(kpis["seasonality_score"].mean(), 2),
})
headline.to_frame("value")

### 5.2 When is the shop busiest?
Share of all clicks per month. The dashed line marks an average month (100 % ÷ 12).
The hover also shows how many categories peak in each month.

In [ ]:
month_plot = month_share.round(1).rename("share").reset_index()
month_plot["month_name"] = month_plot["month"].map(MONTH_NAMES)
month_plot["categories_peaking"] = month_plot["month"].map(peak_counts).fillna(0)

fig = px.bar(month_plot, x="month_name", y="share", hover_data=["categories_peaking"],
             labels={"month_name": "Month", "share": "Share of clicks (%)"},
             title="Share of clicks per month")
fig.add_hline(y=100 / 12, line_dash="dash", annotation_text="average month")
fig.show()

### 5.3 Which categories matter most?
Size against seasonality. The dashed lines are the medians that define the priority groups.
**Top right = big and seasonal**: plan campaigns for these first.

In [ ]:
fig = px.scatter(kpis.reset_index(), x="seasonality_score", y="avg_monthly_clicks",
                 color="priority", hover_name="category", hover_data=["peak_month", "low_month"],
                 color_discrete_map=PRIORITY_COLORS, category_orders={"priority": list(PRIORITY_COLORS)},
                 labels={"seasonality_score": "Seasonality score", "avg_monthly_clicks": "Average monthly clicks"},
                 title="Size vs. seasonality")
fig.add_vline(x=kpis["seasonality_score"].median(), line_dash="dash")
fig.add_hline(y=kpis["avg_monthly_clicks"].median(), line_dash="dash")
fig.show()

### 5.4 Priority list
All categories of the most important group, sorted by size.

In [ ]:
(kpis[kpis["priority"] == "1. big & seasonal"]
 .sort_values("avg_monthly_clicks", ascending=False)
 [["seasonality_score", "peak_month", "low_month", "avg_monthly_clicks", "peak_season"]])

## 6. Top n categories by clicks

### 6.1 Choose n
Change `n` to show more or fewer categories (in the web app this is the slider).

In [ ]:
n = 10

### 6.2 Ranking
Total clicks over all 12 months of 2021, plus the share the top n have of all smartbuy clicks.

In [ ]:
ranking = kpis.copy()
ranking["total_clicks"] = monthly.sum(axis=1)
ranking = ranking.sort_values("total_clicks", ascending=False)
top = ranking.head(n)

print(f"Top {n} categories: {top['total_clicks'].sum():,.0f} clicks "
      f"= {top['total_clicks'].sum() / ranking['total_clicks'].sum():.1%} of all clicks")
top[["total_clicks", "avg_monthly_clicks", "peak_month", "peak_season", "seasonality_score"]]

### 6.3 Top n chart
Colour = the season in which the category peaks.

In [ ]:
fig = px.bar(top.reset_index(), x="total_clicks", y="category", color="peak_season", orientation="h",
             color_discrete_map=SEASON_COLORS, category_orders={"peak_season": SEASON_ORDER},
             labels={"total_clicks": "Total clicks 2021", "category": "", "peak_season": "Peak season"},
             title=f"Top {n} categories by clicks")
fig.update_yaxes(categoryorder="total ascending")
fig.show()

## 7. It's all about seasonality

### 7.1 Heatmap of all categories
Every row is a category, every column a month. Red = above average, blue = below average, white = average month.
Categories are sorted by their peak month, so the seasonal waves become visible.

In [ ]:
order = seasonal.idxmax(axis=1).sort_values().index

plt.figure(figsize=(10, 20))
sns.heatmap(seasonal.loc[order], cmap="RdBu_r", center=1)
plt.title("Seasonality index by category (1 = average month)")
plt.xlabel("Month")
plt.ylabel("")
plt.show()

### 7.2 How many categories peak in each season?

In [ ]:
kpis["peak_season"].value_counts().reindex(SEASON_ORDER).to_frame("categories")

### 7.3 Average pattern per season group
Each line is the mean seasonality index of all categories that peak in that season.

In [ ]:
season_shape = (monthly_long.groupby(["peak_season", "month", "month_name"], as_index=False)["index"].mean())

fig = px.line(season_shape, x="month_name", y="index", color="peak_season", markers=True,
              color_discrete_map=SEASON_COLORS, category_orders={"peak_season": SEASON_ORDER},
              labels={"month_name": "Month", "index": "Seasonality index", "peak_season": "Group"},
              title="Average pattern per season group")
fig.add_hline(y=1, line_dash="dash")
fig.show()

### 7.4 The most seasonal categories
The 15 categories with the highest seasonality score.

In [ ]:
top_seasonal = kpis.nlargest(15, "seasonality_score").reset_index()

fig = px.bar(top_seasonal, x="seasonality_score", y="category", color="peak_season", orientation="h",
             hover_data=["peak_month", "low_month", "peak_vs_low"],
             color_discrete_map=SEASON_COLORS, category_orders={"peak_season": SEASON_ORDER},
             labels={"seasonality_score": "Seasonality score", "category": ""},
             title="Top 15 most seasonal categories")
fig.update_yaxes(categoryorder="total ascending")
fig.show()

### 7.5 Deep dive into one category
Change `category` to look at any other category.

In [ ]:
category = "sun care"
kpis.loc[category]

In [ ]:
fig = px.line(monthly_long[monthly_long["category"] == category], x="month_name", y="clicks", markers=True,
              labels={"month_name": "Month", "clicks": "Clicks"}, title=f"Monthly clicks: {category}")
fig.show()

### 7.6 Similar categories
Categories with a similar monthly pattern have a high **correlation** of their seasonality index.
We show the four most similar ones next to the selected category.

In [ ]:
similar = seasonal.T.corr()[category].drop(category).nlargest(4)
similar.round(3).to_frame("correlation")

In [ ]:
fig = px.line(monthly_long[monthly_long["category"].isin([category] + similar.index.tolist())],
              x="month_name", y="index", color="category", markers=True,
              labels={"month_name": "Month", "index": "Seasonality index"},
              title=f"{category} vs. similar categories")
fig.add_hline(y=1, line_dash="dash")
fig.show()

## 8. Key findings and recommendations

### 8.1 Numbers behind the findings

In [ ]:
top_cat = kpis["seasonality_score"].idxmax()
season_counts = kpis["peak_season"].value_counts()

pd.Series({
    "busiest month": f"{MONTH_NAMES[month_share.idxmax()]} ({month_share.max():.1f}%)",
    "quietest month": f"{MONTH_NAMES[month_share.idxmin()]} ({month_share.min():.1f}%)",
    "most common peak": f"{MONTH_NAMES[peak_counts.idxmax()]} ({peak_counts.max()} categories)",
    "seasonal categories peaking Sep-Dec": kpis.loc[kpis["strength"] != "flat", "peak_month"].between(9, 12).sum(),
    "peak season counts": season_counts.reindex(SEASON_ORDER).to_dict(),
    "most seasonal category": f"{top_cat} (score {kpis.loc[top_cat, 'seasonality_score']:.2f}, "
                              f"{kpis.loc[top_cat, 'peak_vs_low']:.1f}x peak vs. low)",
    "big & seasonal categories": (kpis["priority"] == "1. big & seasonal").sum(),
}).to_frame("value")

### 8.2 Key findings
1. **November is the busiest month** with 10.6 % of all clicks; **February is the quietest** with 7.2 %.
2. **November is also the most common peak month**: 34 categories peak there, and 48 seasonal categories peak between September and December.
3. **Winter (31) and autumn (29) peakers dominate**, far ahead of spring (11) and summer (10); 19 categories have no clear season.
4. **37 categories are strongly seasonal** (score above 1.0).
5. The **most seasonal category is *Ski helmets & snowboard helmets*** (score 2.87), peaking in January with 21.4× the clicks of its lowest month.
6. **25 categories are big & seasonal**: they have the biggest upside for campaign planning.

### 8.3 Recommendations for the Content Department
- **Replace eyeballing** with the seasonality score: anything above 1.0 is strongly seasonal.
- **Start campaigns 4–6 weeks before** a category's peak month, especially before the Q4 peak.
- **Focus on the big & seasonal group first**; it combines size and a clear peak.
- Use **flat categories** as steady, all-year traffic drivers.
- **Refresh the analysis** with each new year of data and explore the results in the web app.